In [ ]:
from pathlib import Path
import os

# Set PROJECT_DATA_DIR before launching Jupyter to use data stored elsewhere.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(PROJECT_ROOT / "data"))).expanduser().resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score


In [ ]:
script_dir = os.getcwd()

csv_path = os.path.join(script_dir, str(DATA_DIR / 'Fraud Detection Transactions Dataset.csv'))

df = pd.read_csv(csv_path)

In [ ]:
df.head(5)

In [ ]:
features = ['Transaction_ID', 'User_ID', 'Transaction_Amount', 'Transaction_Type', 'Timestamp', 'Account_Balance', 'Device_Type', 'Location', 'Merchant_Category', 'IP_Address_Flag', 'Daily_Transaction_Count', 'Avg_Transaction_Amount_7d', 'Failed_Transaction_Count_7d', 'Card_Type', 'Card_Age', 'Transaction_Distance', 'Authentication_Method', 'Risk_Score', 'Is_Weekend']
target = ['Fraud_Label']

In [ ]:
print(df[features].head())

In [ ]:
# Identify categorical columns that need one-hot encoding
categorical_columns = ['Transaction_Type', 'Device_Type', 'Location', 'Merchant_Category', 'Card_Type', 'Authentication_Method']

# Apply one-hot encoding
df_encoded = pd.get_dummies(df, columns=categorical_columns, drop_first=False)

print("Original shape:", df.shape)
print("Encoded shape:", df_encoded.shape)
print("\nNew columns created:")
print(df_encoded.columns.tolist())

In [ ]:
# Drop leakage + unnecessary columns
columns_to_drop = [
    'Transaction_ID',
    'User_ID',
    'Timestamp',
    'Risk_Score'   # if removing leakage
]

X = df_encoded.drop(columns=columns_to_drop + ['Fraud_Label'])
y = df_encoded['Fraud_Label']


In [ ]:
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import train_test_split 
clf = RandomForestClassifier( 
    n_estimators=200, 
    random_state=0, 
    oob_score=True, 
    class_weight='balanced', # VERY important for fraud 
    n_jobs=-1 
    ) 

X_train, X_test, y_train, y_test = train_test_split( 
    X, 
    y, 
    test_size=0.2, 
    stratify=y, # preserve fraud ratio 
    random_state=42 
    )

clf.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score 
y_pred = clf.predict(X_test) 
y_proba = clf.predict_proba(X_test)[:, 1] 
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred)) 
print("\nReport:\n", classification_report(y_test, y_pred)) 
print("ROC-AUC:", roc_auc_score(y_test, y_proba)) 
print("PR-AUC :", average_precision_score(y_test, y_proba))

In [ ]:
import numpy as np

y_true = y_test
y_pred = clf.predict(X_test)

TN = np.sum((y_true == 0) & (y_pred == 0))
FP = np.sum((y_true == 0) & (y_pred == 1))
FN = np.sum((y_true == 1) & (y_pred == 0))
TP = np.sum((y_true == 1) & (y_pred == 1))

print("TN, FP, FN, TP:", TN, FP, FN, TP)


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=[0, 1],   # or your real class names
    values_format='d'
)
plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

proba = clf.predict_proba(X_test)[:, 1]

threshold = 0.22
y_pred = (proba >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

rf = RandomForestClassifier(
    class_weight="balanced",
    random_state=0,
    n_jobs=-1,
    oob_score=False
)

param_dist = {
    "n_estimators": randint(300, 1200),
    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_leaf": randint(1, 30),
    "min_samples_split": randint(2, 40),
    "max_features": ["sqrt", "log2", None]
}

search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=30,
    scoring="average_precision",  # PR-AUC
    cv=3,
    n_jobs=-1,
    random_state=0
)

search.fit(X_train, y_train)
print(search.best_params_)
print("Best CV PR-AUC:", search.best_score_)
best_model = search.best_estimator_
